# Service Access — Silver
**Water • Electricity • Internet**

Clean percentages, use the shared region crosswalk, and insert/update Silver. IDs start at **400000001**. Missing values stay NULL.

**Input:** `pondolens.dev_bronze.service_access`  
**Output:** `pondolens.dev_silver.service_access`

Run in order, with one writer at a time. Set the year range below and configure the shared crosswalk in Step 2 before running the batch.

`value` and `source` remain in Silver to preserve measurements and separate source series. Databricks execution still needs verification.


## Settings
Change only these two numbers to adjust coverage. Valid years outside this range are skipped and reported; they do not fail the batch. Blank, nonnumeric or nonpositive years still block the batch.

The previous 145-row baseline covers 2021–2025; it is a reference, not a fixed required count.


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW service_access_settings AS
SELECT 2021 AS start_year, 2025 AS end_year;


In [0]:
%sql
SELECT ASSERT_TRUE(start_year >= 1 AND end_year >= start_year,
 'Fix the year range in Settings.')
FROM service_access_settings;


## 1. Clean and select
Select drinking-water access, electricity access and internet access at home. Trim whitespace and cast year. The earlier 2021–2025 baseline had 145 selected rows, including missing values.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW service_access_clean_text AS
WITH cleaned AS (
 SELECT
  NULLIF(TRIM(REGEXP_REPLACE(country, r'\s+', ' ')), '') AS country,
  NULLIF(TRIM(year), '') AS year_raw,
  TRY_CAST(year AS INT) AS year,
  NULLIF(TRIM(REGEXP_REPLACE(region, r'\s+', ' ')), '') AS region_raw,
  NULLIF(TRIM(REGEXP_REPLACE(indicator, r'\s+', ' ')), '') AS service_measure,
  NULLIF(TRIM(REGEXP_REPLACE(breakdown_type, r'\s+', ' ')), '') AS breakdown_type,
  NULLIF(TRIM(REGEXP_REPLACE(breakdown_value, r'\s+', ' ')), '') AS breakdown_value,
  NULLIF(TRIM(value), '') AS value_raw,
  NULLIF(TRIM(REGEXP_REPLACE(unit, r'\s+', ' ')), '') AS unit,
  NULLIF(TRIM(REGEXP_REPLACE(source, r'\s+', ' ')), '') AS source,
  NULLIF(TRIM(notes), '') AS notes,
  _rescued_data, _source_file, _run_id, _ingested_at
 FROM pondolens.dev_bronze.service_access
)
SELECT *, CASE
 WHEN LOWER(service_measure) LIKE 'at least basic drinking-water%' THEN 'Water'
 WHEN service_measure = 'Electricity access' THEN 'Electricity'
 WHEN service_measure = 'Internet access at home' THEN 'Internet'
END AS service
FROM cleaned;

In [0]:
%sql
SELECT year, service, service_measure, unit, COUNT(*) AS total_rows
FROM service_access_clean_text WHERE service IS NOT NULL
GROUP BY year, service, service_measure, unit ORDER BY year, service;

## 2. Use the shared region crosswalk
All Silver notebooks must read the same verified mapping. Edit the table name and column aliases below to match your group's crosswalk.

**Required columns:** country, source_region, region_code, region_name, valid_from_year, valid_to_year.

The table name below is a proposed name, not a confirmed existing table. Do not create invented mappings to bypass this step.

- Map aliases such as `IV-A` and `REGION IVA` to the team's canonical CALABARZON code. Verify what `04` represents in each source first.
- Use one agreed national code for `PH` / `PHL`; keep national rows separate from regions.
- Validity years must reflect boundaries applicable to each observation. Do not map historical records to today's boundaries automatically.
- For source-dependent aliases, expose a source-specific adapter to the shared mapping; do not use an ambiguous country/label mapping.

One alias must match exactly one region for a given year. Changing canonical codes in an existing Silver table requires a coordinated migration because the code is part of the match key.


In [0]:
%sql
SELECT DISTINCT country, region_raw, year
FROM service_access_clean_text
WHERE service IS NOT NULL
ORDER BY country, region_raw, year;


In [0]:
%sql
CREATE OR REPLACE TABLE pondolens.dev_silver.shared_region_crosswalk
USING DELTA AS
SELECT 'Philippines' AS country, 'BARMM' AS source_region, '15' AS region_code, 'BARMM' AS region_name, 2021 AS valid_from_year, 2025 AS valid_to_year
UNION ALL SELECT 'Philippines', 'CAR', '14', 'CAR', 2021, 2025
UNION ALL SELECT 'Philippines', 'MIMAROPA', '17', 'MIMAROPA', 2021, 2025
UNION ALL SELECT 'Philippines', 'NCR', '13', 'NCR', 2021, 2025
UNION ALL SELECT 'Philippines', 'National', 'PHL', 'National', 2021, 2025
UNION ALL SELECT 'Philippines', 'Region I (Ilocos Region)', '01', 'Region I (Ilocos Region)', 2021, 2025
UNION ALL SELECT 'Philippines', 'Region II (Cagayan Valley)', '02', 'Region II (Cagayan Valley)', 2021, 2025
UNION ALL SELECT 'Philippines', 'Region III (Central Luzon)', '03', 'Region III (Central Luzon)', 2021, 2025
UNION ALL SELECT 'Philippines', 'Region IV-A (CALABARZON)', '04', 'Region IV-A (CALABARZON)', 2021, 2025
UNION ALL SELECT 'Philippines', 'Region V (Bicol Region)', '05', 'Region V (Bicol Region)', 2021, 2025
UNION ALL SELECT 'Philippines', 'Region VI (Western Visayas)', '06', 'Region VI (Western Visayas)', 2021, 2025
UNION ALL SELECT 'Philippines', 'Region VII (Central Visayas)', '07', 'Region VII (Central Visayas)', 2021, 2025
UNION ALL SELECT 'Philippines', 'Region VIII (Eastern Visayas)', '08', 'Region VIII (Eastern Visayas)', 2021, 2025
UNION ALL SELECT 'Philippines', 'Region IX (Zamboanga Peninsula)', '09', 'Region IX (Zamboanga Peninsula)', 2021, 2025
UNION ALL SELECT 'Philippines', 'Region X (Northern Mindanao)', '10', 'Region X (Northern Mindanao)', 2021, 2025
UNION ALL SELECT 'Philippines', 'Region XI (Davao Region)', '11', 'Region XI (Davao Region)', 2021, 2025
UNION ALL SELECT 'Philippines', 'Region XII (SOCCSKSARGEN)', '12', 'Region XII (SOCCSKSARGEN)', 2021, 2025
UNION ALL SELECT 'Philippines', 'Region XIII (Caraga)', '16', 'Region XIII (Caraga)', 2021, 2025;

CREATE OR REPLACE TEMP VIEW service_access_geography_reference AS
SELECT
 country,
 source_region,
 region_code,
 region_name,
 valid_from_year,
 valid_to_year
FROM pondolens.dev_silver.shared_region_crosswalk;


In [0]:
%sql
SELECT ASSERT_TRUE(COUNT(*) = 0,
 'Crosswalk has overlapping validity periods for the same alias. Fix mapping before continuing.')
FROM (
 SELECT *, MAX(valid_to_year) OVER (
  PARTITION BY LOWER(TRIM(country)), LOWER(TRIM(source_region))
  ORDER BY valid_from_year, valid_to_year, region_code
  ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
 ) AS previous_end_year
 FROM service_access_geography_reference
)
WHERE valid_from_year <= previous_end_year;


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW service_access_expected AS
WITH selected AS (
 SELECT *, TRY_CAST(value_raw AS DECIMAL(18,6)) AS value
 FROM service_access_clean_text CROSS JOIN service_access_settings
 WHERE service IS NOT NULL AND year BETWEEN start_year AND end_year
)
SELECT s.*, NULLIF(TRIM(g.region_name),'') AS region,
 NULLIF(TRIM(g.region_code),'') AS region_code
FROM selected s LEFT JOIN service_access_geography_reference g
 ON LOWER(TRIM(s.country)) = LOWER(TRIM(g.country))
 AND LOWER(TRIM(s.region_raw)) = LOWER(TRIM(g.source_region))
 AND s.year BETWEEN g.valid_from_year AND g.valid_to_year;


## 3. Check before writing
Zero invalid records, duplicate keys and invalid crosswalk rows. `out_of_scope_rows` is informational and does not block a nonempty valid batch. Selected and mapped counts must match. Missing observations are allowed. If a check fails, inspect the records; keep this validation enabled.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW service_access_batch_gate AS
SELECT
 (SELECT COUNT(*) FROM service_access_expected) AS expected_rows,
 (SELECT COUNT(*) FROM service_access_clean_text CROSS JOIN service_access_settings WHERE service IS NOT NULL
   AND year BETWEEN start_year AND end_year) AS selected_rows,
 (SELECT COUNT(*) FROM service_access_clean_text WHERE service IS NOT NULL
   AND (year IS NULL OR year < 1)) AS invalid_years,
 (SELECT COUNT(*) FROM service_access_clean_text CROSS JOIN service_access_settings
  WHERE service IS NOT NULL AND year >= 1
  AND year NOT BETWEEN start_year AND end_year) AS out_of_scope_rows,
 (SELECT COUNT(*) FROM service_access_expected
  WHERE country IS NULL OR LOWER(country) <> 'philippines' OR region_code IS NULL OR region IS NULL
   OR source IS NULL OR unit IS NULL
   OR NULLIF(TRIM(_rescued_data),'') IS NOT NULL
   OR (value_raw IS NOT NULL AND value IS NULL)
   OR (INSTR(unit,'%') > 0 AND (value < 0 OR value > 100))) AS invalid_rows,
 (SELECT COUNT(*) FROM (
  SELECT country, year, region_code, service_measure, breakdown_type,
   breakdown_value, unit, source FROM service_access_expected
  GROUP BY country, year, region_code, service_measure, breakdown_type,
   breakdown_value, unit, source HAVING COUNT(*) > 1)) AS duplicate_keys,
 (SELECT COUNT(*) FROM service_access_geography_reference
 WHERE NULLIF(TRIM(country),'') IS NULL OR NULLIF(TRIM(source_region),'') IS NULL
 OR NULLIF(TRIM(region_code),'') IS NULL OR NULLIF(TRIM(region_name),'') IS NULL
 OR valid_from_year IS NULL OR valid_to_year IS NULL
 OR valid_from_year > valid_to_year) AS invalid_crosswalk;


In [0]:
%sql
SELECT * FROM service_access_batch_gate;

In [0]:
%sql
SELECT ASSERT_TRUE(
 expected_rows > 0 AND expected_rows = selected_rows
 AND invalid_years = 0 AND invalid_rows = 0
 AND duplicate_keys = 0 AND invalid_crosswalk = 0,
 CONCAT('Batch blocked. Invalid records=', invalid_rows,
 ', duplicate keys=', duplicate_keys, ', invalid years=', invalid_years,
 ', invalid crosswalk=', invalid_crosswalk,
 ', mapped rows=', expected_rows, ', selected rows=', selected_rows)
)
FROM service_access_batch_gate;

In [0]:
%sql
SELECT year, region_raw, service_measure, value_raw, unit, source
FROM service_access_expected
WHERE region IS NULL OR region_code IS NULL
 OR source IS NULL OR unit IS NULL
 OR (value_raw IS NOT NULL AND value IS NULL)
 OR (INSTR(unit, '%') > 0 AND (value < 0 OR value > 100));

## 4. Create the target
Create the Delta table if absent. An existing table is kept; this does not migrate its schema. processed_at records Silver processing time in UTC.

In [0]:
%sql
SET TIME ZONE 'UTC';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS pondolens.dev_silver.service_access (
 service_access_id INT, year INT, unit STRING, service STRING,
 service_measure STRING, region STRING, region_code STRING,
 breakdown_type STRING, breakdown_value STRING,
 value DECIMAL(18,6), source STRING, processed_at TIMESTAMP
) USING DELTA;


## 5. Check existing Silver keys
IDs and observation keys must already be unique before MERGE.

In [0]:
%sql
SELECT ASSERT_TRUE(COUNT(*) = COUNT(DISTINCT service_access_id)
 AND COUNT_IF(service_access_id IS NULL OR service_access_id <= 400000000) = 0
 AND COUNT_IF(year IS NULL OR region_code IS NULL OR service_measure IS NULL OR unit IS NULL OR source IS NULL) = 0,
 'Target has invalid or duplicate IDs/required keys')
FROM pondolens.dev_silver.service_access;

In [0]:
%sql
SELECT ASSERT_TRUE(COUNT(*) = 0,'Target has duplicate business keys') FROM (
 SELECT year, region_code, service_measure, breakdown_type, breakdown_value, unit, source
 FROM pondolens.dev_silver.service_access
 GROUP BY year, region_code, service_measure, breakdown_type, breakdown_value, unit, source
 HAVING COUNT(*) > 1);

## 6. Prepare batch IDs
Existing observations retain IDs. New observations receive IDs above the current maximum.
The match key is year + region_code + service_measure + breakdown + unit + source. `<=>` compares nullable fields safely. No automatic deletion or deduplication.


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW service_access_batch_upsert AS
WITH input AS (
 SELECT s.* FROM service_access_expected s CROSS JOIN service_access_batch_gate g
 WHERE CASE WHEN g.expected_rows > 0 AND g.expected_rows = g.selected_rows
 AND g.invalid_years = 0 AND g.invalid_rows = 0 AND g.duplicate_keys = 0 AND g.invalid_crosswalk = 0
 THEN TRUE ELSE CAST(RAISE_ERROR('Batch validation failed') AS BOOLEAN) END
), matched AS (
 SELECT s.*, t.service_access_id AS existing_id
 FROM input s LEFT JOIN pondolens.dev_silver.service_access t ON s.year <=> t.year AND
 s.region_code <=> t.region_code AND
 s.service_measure <=> t.service_measure AND
 s.breakdown_type <=> t.breakdown_type AND
 s.breakdown_value <=> t.breakdown_value AND
 s.unit <=> t.unit AND
 s.source <=> t.source
), new_rows AS (
 SELECT *, ROW_NUMBER() OVER (ORDER BY year, region_code, service_measure,
 breakdown_type, breakdown_value, unit, source) AS new_sequence
 FROM matched WHERE existing_id IS NULL
), max_id AS (
 SELECT COALESCE(MAX(service_access_id),400000000) AS last_id
 FROM pondolens.dev_silver.service_access
)
SELECT existing_id AS service_access_id, year, unit, service, service_measure,
 region, region_code, breakdown_type, breakdown_value, value, source
FROM matched WHERE existing_id IS NOT NULL
UNION ALL
SELECT CASE WHEN CAST(m.last_id AS BIGINT) + s.new_sequence <= 2147483647
 THEN CAST(CAST(m.last_id AS BIGINT) + s.new_sequence AS INT)
 ELSE CAST(RAISE_ERROR('INT ID allocation exhausted') AS INT) END,
 s.year, s.unit, s.service, s.service_measure, s.region, s.region_code,
 s.breakdown_type, s.breakdown_value, s.value, s.source
FROM new_rows s CROSS JOIN max_id m;

## 7. MERGE
Insert new observations; update changed values or labels. Unchanged rows keep their ID and processed_at. This requires one writer at a time.

In [0]:
%sql
MERGE INTO pondolens.dev_silver.service_access t
USING service_access_batch_upsert s
ON s.year <=> t.year AND
 s.region_code <=> t.region_code AND
 s.service_measure <=> t.service_measure AND
 s.breakdown_type <=> t.breakdown_type AND
 s.breakdown_value <=> t.breakdown_value AND
 s.unit <=> t.unit AND
 s.source <=> t.source
WHEN MATCHED AND (
 NOT (t.value <=> s.value) OR NOT (t.region <=> s.region)
 OR NOT (t.service <=> s.service)
) THEN UPDATE SET
 t.value = s.value, t.region = s.region, t.service = s.service,
 t.processed_at = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN INSERT (
 service_access_id, year, unit, service, service_measure, region, region_code,
 breakdown_type, breakdown_value, value, source, processed_at
) VALUES (
 s.service_access_id, s.year, s.unit, s.service, s.service_measure,
 s.region, s.region_code, s.breakdown_type, s.breakdown_value,
 s.value, s.source, CURRENT_TIMESTAMP()
);

## 8. Check results
Counts must MATCH; the record comparison should return zero rows. Reconciliation checks the incoming batch; earlier observations outside the batch are retained.

In [0]:
%sql
WITH counts AS (
 SELECT (SELECT COUNT(*) FROM service_access_expected) AS batch_rows,
 (SELECT COUNT(*) FROM pondolens.dev_silver.service_access t
 WHERE EXISTS (SELECT 1 FROM service_access_expected s WHERE s.year <=> t.year AND
 s.region_code <=> t.region_code AND
 s.service_measure <=> t.service_measure AND
 s.breakdown_type <=> t.breakdown_type AND
 s.breakdown_value <=> t.breakdown_value AND
 s.unit <=> t.unit AND
 s.source <=> t.source)) AS matched_silver_rows
)
SELECT *, CASE WHEN batch_rows = matched_silver_rows THEN 'MATCH' ELSE 'MISMATCH' END AS status FROM counts;

In [0]:
%sql
WITH expected AS (
 SELECT year, unit, service, service_measure, region, region_code,
 breakdown_type, breakdown_value, value, source FROM service_access_expected
), actual AS (
 SELECT year, unit, service, service_measure, region, region_code,
 breakdown_type, breakdown_value, value, source FROM pondolens.dev_silver.service_access t
 WHERE EXISTS (SELECT 1 FROM service_access_expected s WHERE s.year <=> t.year AND
 s.region_code <=> t.region_code AND
 s.service_measure <=> t.service_measure AND
 s.breakdown_type <=> t.breakdown_type AND
 s.breakdown_value <=> t.breakdown_value AND
 s.unit <=> t.unit AND
 s.source <=> t.source)
), missing AS (SELECT * FROM expected EXCEPT ALL SELECT * FROM actual),
extra AS (SELECT * FROM actual EXCEPT ALL SELECT * FROM expected)
SELECT 'MISSING_OR_CHANGED' AS issue, * FROM missing
UNION ALL SELECT 'EXTRA_OR_CHANGED' AS issue, * FROM extra;

In [0]:
%sql
SELECT service, year, COUNT(*) AS rows,
 COUNT(value) AS observed_values,
 COUNT(*) - COUNT(value) AS missing_values
FROM pondolens.dev_silver.service_access
GROUP BY service, year ORDER BY service, year;

## 9. Verify rerun
Rerun unchanged input: no inserted/updated records, and existing IDs/processed_at stay unchanged. Inspect MERGE operation metrics below. If metrics are absent, compare before/after snapshots.
**BP/AP3:** regional access can provide needs context when geography, years, units and source definitions are compatible. National electricity observations must not be repeated as regional data. Do not sum percentages. AP1/AP2 need other spending/execution tables.


In [0]:
%sql
DESCRIBE HISTORY pondolens.dev_silver.service_access LIMIT 3;